# UTAU 母音遷移分析

`oto.ini` から抽出した VCV 遷移タイミングを確認するためのノートブック。`transition_ms = preutterance_ms - overlap_ms` を、Web プロトタイプの母音補間時間の初期値として参照する。

In [ ]:
import csv
import json
from collections import Counter
from pathlib import Path
from statistics import mean, median

import matplotlib.pyplot as plt

cwd = Path.cwd()
if cwd.name == "notebooks":
    ROOT = cwd.parent
elif (cwd / "data" / "processed").exists():
    ROOT = cwd
else:
    ROOT = cwd / "research"

analysis_dir = ROOT / "data" / "processed" / "analysis"
exports_dir = ROOT / "data" / "processed" / "exports"
transitions_path = analysis_dir / "utau-vowel-transitions.csv"
curves_path = exports_dir / "vowel-transition-curves.generated.json"

with transitions_path.open(encoding="utf-8", newline="") as f:
    transitions = list(csv.DictReader(f))
curves = json.loads(curves_path.read_text(encoding="utf-8"))

len(transitions), curves["metadata"]

In [ ]:
values = [float(row["transition_ms"]) for row in transitions]
{
    "count": len(values),
    "median_ms": median(values),
    "mean_ms": mean(values),
    "min_ms": min(values),
    "max_ms": max(values),
}

In [ ]:
Counter(row["transition"] for row in transitions).most_common()

In [ ]:
pairs = sorted(curves["transitions"].items())
labels = [key for key, _ in pairs]
medians = [value["medianTransitionMs"] for _, value in pairs]

plt.figure(figsize=(12, 5))
plt.bar(labels, medians)
plt.axhline(curves["metadata"]["recommendedDefaultTransitionMs"], color="black", linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.ylabel("transition ms")
plt.title("Median VCV transition time by vowel pair")
plt.tight_layout();